## 04_modeling: Dự đoán supervised
Mẫu train regression/forecasting với RandomForest và LightGBM.

In [1]:
import os
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Load dữ liệu đã xử lý
candidates = [
    '../data/processed/weather_featured.csv',
    '../data/processed/weather_cleaned.csv',
    '../data/processed/weather_mining.csv'
]

data_path = None
for p in candidates:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError(f"Không tìm thấy file data trong {candidates}")

print(f"Đang dùng file: {data_path}")

df = pd.read_csv(data_path)

# Tự động xác định cột thời gian nếu có
time_cols = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
if len(time_cols) > 0:
    time_col = time_cols[0]
    df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
    df = df.sort_values(time_col).reset_index(drop=True)
else:
    time_col = None

# Xác định cột mục tiêu (target) cho bài toán dự báo
target_candidates = ['target_variable', 'Temperature (C)', 'temp', 'value']
selected = next((c for c in target_candidates if c in df.columns), None)
if selected is None:
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if not numeric_cols:
        raise KeyError("Không tìm thấy cột số để làm mục tiêu (target)")
    selected = numeric_cols[0]
    print(f"(Auto) Chọn {selected} làm target vì là cột số đầu tiên.")

print(f"Sử dụng cột mục tiêu: {selected}")
y = df[selected]

# Loại bỏ cột thời gian và mục tiêu khỏi tập đặc trưng
drop_cols = [selected]
if time_col is not None and time_col in df.columns:
    drop_cols.append(time_col)
X = df.drop(columns=drop_cols)

# Chỉ sử dụng các feature số để chạy mô hình (loại bỏ chuỗi/chi tiết không phù hợp)
X = X.select_dtypes(include=["number"])
# Thay thế giá trị thiếu nếu có
X = X.ffill().bfill().fillna(0)

# time series split
tscv = TimeSeriesSplit(n_splits=5)

# Random Forest (regression)
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
start = time.time()
rf_scores = cross_val_score(rf, X, y, cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1)
duration_rf = time.time() - start
print(f"RF (n_estimators=100) RMSE CV: {-rf_scores.mean():.4f} (cv time {duration_rf:.1f}s)")

# LightGBM (regression)
lgb_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
start = time.time()
lgb_scores = cross_val_score(lgb_model, X, y, cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1)
duration_lgb = time.time() - start
print(f"LGBM (n_estimators=200, lr=0.05) RMSE CV: {-lgb_scores.mean():.4f} (cv time {duration_lgb:.1f}s)")

# Fit final model and evaluate on last split
train_idx, test_idx = list(tscv.split(X))[-1]
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
start = time.time()
lgb_model.fit(X_train, y_train)
duration_fit = time.time() - start

y_pred = lgb_model.predict(X_test)
print(f"Final fit time: {duration_fit:.1f}s")
print('Final RMSE', np.sqrt(mean_squared_error(y_test, y_pred)))
print('Final MAE', mean_absolute_error(y_test, y_pred))

Đang dùng file: ../data/processed/weather_featured.csv


Sử dụng cột mục tiêu: Temperature (C)


RF (n_estimators=100) RMSE CV: 0.0870 (cv time 20.4s)


LGBM (n_estimators=200, lr=0.05) RMSE CV: 0.1198 (cv time 17.3s)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002060 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1854
[LightGBM] [Info] Number of data points in the train set: 46675, number of used features: 11
[LightGBM] [Info] Start training from score 17.748664


Final fit time: 7.8s
Final RMSE 0.07130505205468561
Final MAE 0.040893198029344636


## 05 Khai phá tri thức (Pattern mining / Clustering / Anomaly)

Trong phần này, ta thực hiện các bước khai phá tri thức trên cùng dữ liệu đã chuẩn bị:

- **Clustering**: phân nhóm quan sát để tìm cấu trúc ẩn.
- **Anomaly detection**: phát hiện các quan sát bất thường.
- **Association rules**: khai thác luật kết hợp giữa các đặc trưng (dựa trên Apriori).

Ta đặt các thí nghiệm và ghi lại cấu hình (hyperparams) và thời gian train để thuận tiện so sánh.

In [2]:
import time
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest
from mlxtend.frequent_patterns import apriori, association_rules

# Sử dụng dữ liệu đã tải và chuẩn bị ở trên: X, y
# Nếu dữ liệu quá lớn, chỉ lấy mẫu để chạy nhanh (có thể bỏ dòng này nếu đủ tài nguyên).
X_mining = X.copy()

# --- 1) Clustering ---
print("--- Clustering (KMeans) ---")
start = time.time()
cluster_model = KMeans(n_clusters=5, random_state=42, n_init=10)
cluster_labels = cluster_model.fit_predict(X_mining)
duration = time.time() - start

sil_score = silhouette_score(X_mining, cluster_labels)
print(f"KMeans n_clusters=5, n_init=10 -> silhouette_score={sil_score:.4f} (train {duration:.1f}s)")
print("Cluster sizes:", np.bincount(cluster_labels))

# In ra một số đặc trưng trung bình theo cluster để quan sát pattern
cluster_summary = X_mining.assign(cluster=cluster_labels).groupby('cluster').mean().T
print(cluster_summary.iloc[:min(10, len(cluster_summary))])

# --- 2) Anomaly detection ---
print("\n--- Anomaly Detection (IsolationForest) ---")
start = time.time()
iso = IsolationForest(n_estimators=200, contamination=0.02, random_state=42, n_jobs=-1)
iso.fit(X_mining)
iso_scores = iso.decision_function(X_mining)
iso_pred = iso.predict(X_mining)  # -1 = outlier, 1 = inlier
anomaly_fraction = (iso_pred == -1).mean()
duration = time.time() - start
print(f"IsolationForest contamination=0.02 -> detected fraction={anomaly_fraction:.4f} (train {duration:.1f}s)")
print("Anomaly score summary (min/max):", iso_scores.min(), iso_scores.max())

# Show top anomalous points
anomaly_idx = np.where(iso_pred == -1)[0][:10]
print("First 10 detected anomalies (index in dataset):", anomaly_idx)

# --- 3) Association rule mining (Apriori) ---
print("\n--- Association Rule Mining (Apriori) ---")

# Chuyển đổi một số feature định lượng sang dạng nhãn (binning) để khai thác luật
# Ở đây ta sử dụng 4 bins (quartiles) cho mỗi biến để tạo các "items".
bins = 4
X_binned = X_mining.copy()
for col in X_binned.columns:
    if np.issubdtype(X_binned[col].dtype, np.number):
        X_binned[col] = pd.qcut(X_binned[col].rank(method='first'), q=bins, labels=[f"{col}_q{i+1}" for i in range(bins)])

# one-hot encode các item
X_items = pd.get_dummies(X_binned.astype(str))

# Apriori
start = time.time()
frequent = apriori(X_items, min_support=0.05, use_colnames=True)
duration = time.time() - start
print(f"Apriori frequent itemsets (min_support=0.05): {len(frequent)} itemsets (train {duration:.1f}s)")

# Tạo luật kết hợp (association rules)
rules = association_rules(frequent, metric="confidence", min_threshold=0.6)
print(f"Generated {len(rules)} rules (min_confidence=0.6)")

# Hiển thị một số luật đầu tiên
if len(rules) > 0:
    display(rules.sort_values('lift', ascending=False).head(10))
else:
    print("Không có luật nào thỏa điều kiện.")

--- Clustering (KMeans) ---


KMeans n_clusters=5, n_init=10 -> silhouette_score=0.5348 (train 0.5s)
Cluster sizes: [18517 14748   572  9819 12354]
cluster                             0            1            2            3  \
Apparent Temperature (C)    16.747162    18.628923    19.027525    18.488217   
Humidity                     0.699555     0.658175     0.648531     0.664935   
Wind Speed (km/h)           11.554838     9.520016    11.768256     9.525543   
Wind Bearing (degrees)     317.786953   143.968335   195.807692   231.076281   
Visibility (km)             11.721775    11.954954     9.537083    11.717101   
Loud Cover                   0.000000     0.000000     0.000000     0.000000   
Pressure (millibars)      1015.628703  1014.696680     0.000000  1013.814754   
year                      2011.031377  2010.952739  2009.368881  2010.886954   
month                        6.893503     6.932398     7.295455     6.624096   
hour                        11.357779    11.282208    11.395105    11.963031   

c

IsolationForest contamination=0.02 -> detected fraction=0.0200 (train 2.1s)
Anomaly score summary (min/max): -0.09728370162089361 0.16707125178744975
First 10 detected anomalies (index in dataset): [  0   1   3 171 219 244 246 255 292 376]

--- Association Rule Mining (Apriori) ---


Apriori frequent itemsets (min_support=0.05): 1322 itemsets (train 2.9s)
Generated 1105 rules (min_confidence=0.6)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
897,(Apparent Temperature (C)_Apparent Temperature...,"(Loud Cover_Loud Cover_q2, temp_lag1_temp_lag1...",0.056811,0.056829,0.050509,0.889063,15.644501,1.0,0.047280,8.501897,0.992463,0.800057,0.882379,0.888924
895,"(Loud Cover_Loud Cover_q2, Apparent Temperatur...","(year_year_q2, temp_lag1_temp_lag1_q4)",0.056811,0.056829,0.050509,0.889063,15.644501,1.0,0.047280,8.501897,0.992463,0.800057,0.882379,0.888924
894,"(Loud Cover_Loud Cover_q2, temp_lag1_temp_lag1...",(Apparent Temperature (C)_Apparent Temperature...,0.056829,0.056811,0.050509,0.888784,15.644501,1.0,0.047280,8.480705,0.992482,0.800057,0.882085,0.888924
896,"(year_year_q2, temp_lag1_temp_lag1_q4)","(Loud Cover_Loud Cover_q2, Apparent Temperatur...",0.056829,0.056811,0.050509,0.888784,15.644501,1.0,0.047280,8.480705,0.992482,0.800057,0.882085,0.888924
889,"(year_year_q1, temp_lag1_temp_lag1_q4)","(Loud Cover_Loud Cover_q1, Apparent Temperatur...",0.061578,0.061596,0.055151,0.895622,14.540227,1.0,0.051358,8.990430,0.992331,0.810761,0.888771,0.895492
887,"(Loud Cover_Loud Cover_q1, temp_lag1_temp_lag1...","(year_year_q1, Apparent Temperature (C)_Appare...",0.061578,0.061596,0.055151,0.895622,14.540227,1.0,0.051358,8.990430,0.992331,0.810761,0.888771,0.895492
886,"(Loud Cover_Loud Cover_q1, Apparent Temperatur...","(year_year_q1, temp_lag1_temp_lag1_q4)",0.061596,0.061578,0.055151,0.895362,14.540227,1.0,0.051358,8.968296,0.992350,0.810761,0.888496,0.895492
888,"(year_year_q1, Apparent Temperature (C)_Appare...","(Loud Cover_Loud Cover_q1, temp_lag1_temp_lag1...",0.061596,0.061578,0.055151,0.895362,14.540227,1.0,0.051358,8.968296,0.992350,0.810761,0.888496,0.895492
744,(Apparent Temperature (C)_Apparent Temperature...,"(temp_lag1_temp_lag1_q1, year_year_q3)",0.062864,0.062882,0.056901,0.905141,14.394357,1.0,0.052948,9.879023,0.992949,0.826504,0.898775,0.905012
743,(Apparent Temperature (C)_Apparent Temperature...,"(temp_lag1_temp_lag1_q1, Loud Cover_Loud Cover...",0.062864,0.062882,0.056901,0.905141,14.394357,1.0,0.052948,9.879023,0.992949,0.826504,0.898775,0.905012
